In [ ]:
import zipfile

zip_path = "/content/UCI HAR Dataset.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content")

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score

# Load training data
X_train = np.loadtxt("/content/UCI HAR Dataset/train/X_train.txt")
y_train = np.loadtxt("/content/UCI HAR Dataset/train/y_train.txt")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

accuracies = []

for i in range(50):

    kmeans = KMeans(
        n_clusters=6,
        n_init=1,
        random_state=i
    )

    clusters = kmeans.fit_predict(X_train)

    # Map each cluster to its majority activity
    cluster_to_label = {}

    for cluster in range(6):
        labels = y_train[clusters == cluster]

        if len(labels) > 0:
            values, counts = np.unique(labels, return_counts=True)
            cluster_to_label[cluster] = values[np.argmax(counts)]

    # Convert clusters to activity predictions
    predicted = np.array([
        cluster_to_label[c] for c in clusters
    ])

    accuracy = accuracy_score(y_train, predicted)
    accuracies.append(accuracy)

print("\nNumber of trials:", len(accuracies))
print("Average accuracy:", np.mean(accuracies) * 100, "%")
print("Best accuracy:", np.max(accuracies) * 100, "%")
print("Worst accuracy:", np.min(accuracies) * 100, "%")

X_train shape: (7352, 561)
y_train shape: (7352,)

Number of trials: 50
Average accuracy: 59.35963003264418 %
Best accuracy: 69.43688792165396 %
Worst accuracy: 47.701305767138194 %


In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix

# Load data
X_train = np.loadtxt("/content/UCI HAR Dataset/train/X_train.txt")
y_train = np.loadtxt("/content/UCI HAR Dataset/train/y_train.txt")

# Convert 6 activities into 2 classes
# 1 = Motion, 0 = Sedentary
y_binary = np.where(y_train <= 3, 1, 0)

# K-means with 2 clusters
kmeans = KMeans(
    n_clusters=2,
    n_init=10,
    random_state=42
)

clusters = kmeans.fit_predict(X_train)

# Map clusters to actual classes using majority class
cluster_to_label = {}

for cluster in range(2):
    labels = y_binary[clusters == cluster]
    values, counts = np.unique(labels, return_counts=True)
    cluster_to_label[cluster] = values[np.argmax(counts)]

predicted = np.array([
    cluster_to_label[c] for c in clusters
])

accuracy = accuracy_score(y_binary, predicted)

print("Cluster to class mapping:")
print(cluster_to_label)

print("\nBinary K-means accuracy:", accuracy * 100, "%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_binary, predicted))

Cluster to class mapping:
{0: np.int64(0), 1: np.int64(1)}

Binary K-means accuracy: 99.83677910772579 %

Confusion Matrix:
[[4055   12]
 [   0 3285]]


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

X_train = np.loadtxt("/content/UCI HAR Dataset/train/X_train.txt")
y_train = np.loadtxt("/content/UCI HAR Dataset/train/y_train.txt")

X_test = np.loadtxt("/content/UCI HAR Dataset/test/X_test.txt")
y_test = np.loadtxt("/content/UCI HAR Dataset/test/y_test.txt")

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (7352, 561)
Testing: (2947, 561)


In [ ]:
base_train = []
base_test = []

for activity in range(1, 7):

    # Activity vs all other activities
    y_binary_train = np.where(y_train == activity, 1, 0)
    y_binary_test = np.where(y_test == activity, 1, 0)

    svm = SVC(kernel="linear")

    svm.fit(X_train, y_binary_train)

    train_output = svm.decision_function(X_train)
    test_output = svm.decision_function(X_test)

    base_train.append(train_output)
    base_test.append(test_output)

    print("Base SVM", activity, "completed")

base_train = np.array(base_train).T
base_test = np.array(base_test).T

print("\nBase SVM training output shape:", base_train.shape)
print("Base SVM testing output shape:", base_test.shape)

Base SVM 1 completed
Base SVM 2 completed
Base SVM 3 completed
Base SVM 4 completed
Base SVM 5 completed
Base SVM 6 completed

Base SVM training output shape: (7352, 6)
Base SVM testing output shape: (2947, 6)


In [ ]:
# Train the upper SVM using the outputs of the six base SVMs

upper_svm = SVC(kernel="linear")

upper_svm.fit(base_train, y_train)

# Predict activities
y_pred = upper_svm.predict(base_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Multi-layer SVM accuracy:", accuracy * 100, "%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Multi-layer SVM accuracy: 96.60671869697998 %

Confusion Matrix:
[[496   0   0   0   0   0]
 [ 20 450   1   0   0   0]
 [  2   5 413   0   0   0]
 [  0   1   0 434  56   0]
 [  1   0   0  14 517   0]
 [  0   0   0   0   0 537]]

Classification Report:
              precision    recall  f1-score   support

         1.0       0.96      1.00      0.98       496
         2.0       0.99      0.96      0.97       471
         3.0       1.00      0.98      0.99       420
         4.0       0.97      0.88      0.92       491
         5.0       0.90      0.97      0.94       532
         6.0       1.00      1.00      1.00       537

    accuracy                           0.97      2947
   macro avg       0.97      0.97      0.97      2947
weighted avg       0.97      0.97      0.97      2947



In [ ]:
# Plurality voting using the six base SVMs

base_predictions = []

for activity in range(6):
    predictions = np.where(base_test[:, activity] > 0, 1, 0)
    base_predictions.append(predictions)

base_predictions = np.array(base_predictions).T

# Count votes for each activity
plurality_pred = []

for i in range(len(X_test)):
    votes = np.sum(base_predictions[i], axis=0)

    if np.sum(base_predictions[i]) == 1:
        plurality_pred.append(np.argmax(base_predictions[i]) + 1)
    else:
        scores = base_test[i]
        plurality_pred.append(np.argmax(scores) + 1)

plurality_pred = np.array(plurality_pred)

accuracy = accuracy_score(y_test, plurality_pred)

print("Plurality voting accuracy:", accuracy * 100, "%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, plurality_pred))

print("\nClassification Report:")
print(classification_report(y_test, plurality_pred))

Plurality voting accuracy: 96.47098744485918 %

Confusion Matrix:
[[496   0   0   0   0   0]
 [ 19 451   1   0   0   0]
 [  3   5 412   0   0   0]
 [  0   4   0 427  59   1]
 [  1   0   0  11 520   0]
 [  0   0   0   0   0 537]]

Classification Report:
              precision    recall  f1-score   support

         1.0       0.96      1.00      0.98       496
         2.0       0.98      0.96      0.97       471
         3.0       1.00      0.98      0.99       420
         4.0       0.97      0.87      0.92       491
         5.0       0.90      0.98      0.94       532
         6.0       1.00      1.00      1.00       537

    accuracy                           0.96      2947
   macro avg       0.97      0.96      0.96      2947
weighted avg       0.97      0.96      0.96      2947

